In [1]:
from google.colab import drive
import os
import json
import glob

In [2]:
# 1. caching google colab
drive.mount('/content/drive')

drive_log_dir = "/content/drive/MyDrive/VPR_cosplace_logs"
os.makedirs(drive_log_dir, exist_ok=True)

# 2. Clone repo
!git clone --recursive https://github.com/AxelBadouel/Visual-Place-Recognition-Project.git
%cd Visual-Place-Recognition-Project

# 3. Install dependencies and requirements
!pip install -q -r requirements.txt
!pip install -q -r image-matching-models/requirements.txt
!pip install -q faiss-cpu

# 4. Download data
!python download_datasets.py

Mounted at /content/drive
Cloning into 'Visual-Place-Recognition-Project'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 156 (delta 32), reused 24 (delta 17), pack-reused 95 (from 3)
Receiving objects: 100% (156/156), 1.37 MiB | 3.74 MiB/s, done.
Resolving deltas: 100% (35/35), done.
Submodule 'image-matching-models' (https://github.com/alexstoken/image-matching-models.git) registered for path 'image-matching-models'
Cloning into '/content/Visual-Place-Recognition-Project/image-matching-models'...
remote: Enumerating objects: 2853, done.        
remote: Counting objects: 100% (1256/1256), done.        
remote: Compressing objects: 100% (380/380), done.        
remote: Total 2853 (delta 992), reused 892 (delta 874), pack-reused 1597 (from 1)        
Receiving objects: 100% (2853/2853), 84.39 MiB | 26.45 MiB/s, done.
Resolving deltas: 100% (2021/2021), done.
Submodule path 'image-matc

In [3]:
# Verify mapping file
mapping_file = f"{drive_log_dir}/dataset_run_mapping.json"
if os.path.exists(mapping_file):
    with open(mapping_file, "r") as f:
        mapping = json.load(f)
    print("\n[SUCCESS] Loaded existing dataset mapping from Drive:")
    print(json.dumps(mapping, indent=2))
else:
    print("\n[WARNING] No mapping file found. Check your Drive path.")


[SUCCESS] Loaded existing dataset mapping from Drive:
{
  "sf_xs": "/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-14-23",
  "tokyo_xs": "/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-37-08",
  "svox_night": "/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55",
  "svox_sun": "/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-47-17"
}


# 1. VPR Evaluation

In [ ]:
# 5. dataset splits
test_set_db_queries = [
    ("sf_xs",       "/content/Visual-Place-Recognition-Project/data/sf_xs/test/database",       "/content/Visual-Place-Recognition-Project/data/sf_xs/test/queries"),
    ("tokyo_xs",    "/content/Visual-Place-Recognition-Project/data/tokyo_xs/test/database",    "/content/Visual-Place-Recognition-Project/data/tokyo_xs/test/queries"),
    ("svox_night",  "/content/Visual-Place-Recognition-Project/data/svox/images/test/gallery",  "/content/Visual-Place-Recognition-Project/data/svox/images/test/queries_night"),
    ("svox_sun",    "/content/Visual-Place-Recognition-Project/data/svox/images/test/gallery",  "/content/Visual-Place-Recognition-Project/data/svox/images/test/queries_sun"),
]

# Track run directories across sessions via an explicit persistent metadata mapping
drive_log_dir = "/content/drive/MyDrive/VPR_cosplace_logs"
mapping_file = f"{drive_log_dir}/dataset_run_mapping.json"
mapping = json.load(open(mapping_file)) if os.path.exists(mapping_file) else {}

for name, database, queries in test_set_db_queries:
    # Check if Stage 1 global extraction and nearest-neighbor search already exist
    if name in mapping and os.path.isdir(f"{mapping[name]}/preds") and len(os.listdir(f"{mapping[name]}/preds")) > 0:
        print(f"[STATUS] Skipping Stage 1 for {name}: predictions already cached at {mapping[name]}/preds")
        continue

    print(f"\n==================================================================")
    print(f"Executing CosPlace Retrieval on Benchmark Dataset: {name}")
    print(f"==================================================================")

    before_runs = set(glob.glob(f"{drive_log_dir}/*"))

    # Execute global feature extraction and save Top-20 nearest neighbor candidates
    !python VPR-methods-evaluation/main.py \
        --num_workers 4 \
        --batch_size 32 \
        --log_dir {drive_log_dir} \
        --method=cosplace \
        --backbone=ResNet18 \
        --descriptors_dimension=512 \
        --image_size 512 512 \
        --database_folder {database} \
        --queries_folder {queries} \
        --num_preds_to_save 20 \
        --recall_values 1 5 10 20 \
        --distance_metric "L2"

    # update tracking record
    after_runs = set(glob.glob(f"{drive_log_dir}/*"))
    new_runs = after_runs - before_runs
    if new_runs:
        mapping[name] = sorted(new_runs)[-1]
        with open(mapping_file, "w") as f:
            json.dump(mapping, f, indent=2)
        print(f"[METADATA] Successfully mapped dataset '{name}' -> {mapping[name]}")

print("\n[SUMMARY] Active Dataset-to-Directory Mapping:")
print(json.dumps(mapping, indent=2))


Executing CosPlace Retrieval on Benchmark Dataset: sf_xs
2026-08-19 15:14:23 VPR-methods-evaluation/main.py --num_workers 4 --batch_size 32 --log_dir /content/drive/MyDrive/VPR_cosplace_logs --method=cosplace --backbone=ResNet18 --descriptors_dimension=512 --image_size 512 512 --database_folder /content/Visual-Place-Recognition-Project/data/sf_xs/test/database --queries_folder /content/Visual-Place-Recognition-Project/data/sf_xs/test/queries --num_preds_to_save 20 --recall_values 1 5 10 20 --distance_metric L2
2026-08-19 15:14:23 Arguments: Namespace(distance_metric=['L2'], positive_dist_threshold=25, method='cosplace', backbone='ResNet18', descriptors_dimension=512, database_folder=['/content/Visual-Place-Recognition-Project/data/sf_xs/test/database'], queries_folder=['/content/Visual-Place-Recognition-Project/data/sf_xs/test/queries'], num_workers=4, batch_size=32, log_dir='/content/drive/MyDrive/VPR_cosplace_logs', device='cuda', recall_values=[1, 5, 10, 20], no_labels=False, num_p

# 2. Image Matching & Re-Ranking on Retrieval Results

In [4]:
import json
import os

drive_log_dir = "/content/drive/MyDrive/VPR_cosplace_logs"
mapping_file = f"{drive_log_dir}/dataset_run_mapping.json"

if not os.path.exists(mapping_file):
    raise FileNotFoundError(f"Mapping file not found at {mapping_file}. Run Cell 1 first.")

with open(mapping_file, "r") as f:
    mapping = json.load(f)

# Evaluate both sparse detector-based and dense detector-free matchers
matchers = ['superpoint-lg', 'superglue', 'loftr']

for name, log_dir in mapping.items():
    preds_dir = f"{log_dir}/preds"
    if not os.path.isdir(preds_dir):
        print(f"[WARNING] Missing Stage 1 candidates for {name} at {preds_dir}. Skipping.")
        continue

    for matcher in matchers:
        # Matcher outputs are stored alongside Stage 1 predictions
        target_inliers_folder = f"{preds_dir}_{matcher}"

        # Resume mechanism: Skip matchers that have already generated inliers
        if os.path.exists(target_inliers_folder) and len(os.listdir(target_inliers_folder)) > 0:
            print(f"[STATUS] Skipping {matcher.upper()} on {name}: inliers already computed.")
            continue

        print(f"\n==================================================================")
        print(f"STAGE 2 MATCHING: Running {matcher.upper()} on Dataset: {name}")
        print(f"Target Predictions Directory: {preds_dir}")
        print(f"==================================================================")

        !python match_queries_preds.py \
            --preds-dir {preds_dir} \
            --matcher {matcher} \
            --device 'cuda' \
            --num-preds 20

[STATUS] Skipping SUPERPOINT-LG on sf_xs: inliers already computed.
[STATUS] Skipping SUPERGLUE on sf_xs: inliers already computed.
[STATUS] Skipping LOFTR on sf_xs: inliers already computed.
[STATUS] Skipping SUPERPOINT-LG on tokyo_xs: inliers already computed.
[STATUS] Skipping SUPERGLUE on tokyo_xs: inliers already computed.
[STATUS] Skipping LOFTR on tokyo_xs: inliers already computed.
[STATUS] Skipping SUPERPOINT-LG on svox_night: inliers already computed.
[STATUS] Skipping SUPERGLUE on svox_night: inliers already computed.

STAGE 2 MATCHING: Running LOFTR on Dataset: svox_night
Target Predictions Directory: /content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55/preds
/content/Visual-Place-Recognition-Project/image-matching-models/matching/third_party/LightGlue/lightglue/lightglue.py:24: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.floa

In [5]:
import json
import os

mapping_file = f"{drive_log_dir}/dataset_run_mapping.json"
with open(mapping_file, "r") as f:
    mapping = json.load(f)

for name, log_dir in mapping.items():
    preds_dir = f"{log_dir}/preds"
    inliers_dirs = f"{preds_dir}_loftr {preds_dir}_superglue {preds_dir}_superpoint-lg"

    print(f"\n==================================================================")
    print(f"FINAL RE-RANKING SCORES: {name.upper()}")
    print(f"Base Directory: {log_dir}")
    print(f"==================================================================")

    !python reranking.py \
        --preds_dir {preds_dir} \
        --inliers_dir {inliers_dirs} \
        --num-preds 20 \
        --recall-values 1 5 10 20


FINAL RE-RANKING SCORES: SF_XS
Base Directory: /content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-14-23
folder name: /content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-14-23/preds
Folder has something in it!
100% 1000/1000 [08:39<00:00,  1.92it/s]
R@1: 77.4, R@5: 79.7, R@10: 80.6, R@20: 81.4
Saved results to /content/drive/MyDrive/VPR_preds/logs/Rerankings/_loftr_recalls.xlsx
folder name: /content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-14-23/preds
Folder has something in it!
100% 1000/1000 [07:44<00:00,  2.15it/s]
R@1: 76.8, R@5: 79.8, R@10: 80.8, R@20: 81.4
Saved results to /content/drive/MyDrive/VPR_preds/logs/Rerankings/_superglue_recalls.xlsx
folder name: /content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-14-23/preds
Folder has something in it!
100% 1000/1000 [12:57<00:00,  1.29it/s]
R@1: 77.3, R@5: 80.2, R@10: 80.9, R@20: 81.4
Saved results to /content/drive/MyDrive/VPR_preds/logs/Rerankings/_superpoint-lg_recalls.xlsx

FINAL RE-RANKING SCORES: TOKYO_XS
Base Di

In [11]:
!python /content/Visual-Place-Recognition-Project/reranking.py \
  --preds_dir "/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55/preds" \
  --inliers_dir "/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55/preds_loftr"

folder name: /content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55/preds
Folder has something in it!
100% 823/823 [00:05<00:00, 161.65it/s]
R@1: 61.4, R@5: 65.4, R@10: 66.2, R@20: 67.6, R@100: 67.6
Saved results to /content/drive/MyDrive/VPR_preds/logs/Rerankings/_loftr_recalls.xlsx


In [12]:
!python /content/Visual-Place-Recognition-Project/reranking.py \
  --preds_dir "/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55/preds" \
  --inliers_dir "/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55/preds_superglue"

folder name: /content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55/preds
Folder has something in it!
100% 823/823 [00:05<00:00, 153.17it/s]
R@1: 59.4, R@5: 64.9, R@10: 66.3, R@20: 67.6, R@100: 67.6
Saved results to /content/drive/MyDrive/VPR_preds/logs/Rerankings/_superglue_recalls.xlsx


In [13]:
!python /content/Visual-Place-Recognition-Project/reranking.py \
  --preds_dir "/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55/preds" \
  --inliers_dir "/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55/preds_superpoint-lg"

folder name: /content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55/preds
Folder has something in it!
100% 823/823 [12:47<00:00,  1.07it/s]
R@1: 60.5, R@5: 65.2, R@10: 66.3, R@20: 67.6, R@100: 67.6
Saved results to /content/drive/MyDrive/VPR_preds/logs/Rerankings/_superpoint-lg_recalls.xlsx
